# Linear Mixed-Effects Model (LMM)
## Fixed Effects of Visual Attention on Perceived Safety

| | |
|---|---|
| **N** | 1,500 observaciones (30 participantes × 50 imágenes) |
| **Dependiente** | score de seguridad percibida (1–10) |
| **Predictores** | tiempo de atención (seg) por clase semántica, estandarizados (z-score) |
| **Efecto aleatorio** | intercepto por participante `(1\|participante)` |
| **Estimación** | REML |
| **Clasificaciones** | ADE20K base · Grouped · Disorder · Grouped+Disorder |

In [1]:
import pandas as pd
import numpy as np
import json
import warnings
warnings.filterwarnings('ignore')
import statsmodels.formula.api as smf
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

print('Librerías OK')

Librerías OK


In [2]:
CSV_PATH    = '../csv/df_final1.csv'
SCORES_PATH = '../json/data_hololens.json'

df = pd.read_csv(CSV_PATH)
with open(SCORES_PATH) as f:
    scores_json = json.load(f)

print(f'Filas CSV:          {len(df):,}')
print(f'Participantes:      {df["participante"].nunique()}')
print(f'Imágenes únicas:    {df["ImageName"].nunique()}')
print(f'Clases ADE20K:      {df["main_class"].nunique()}')

Filas CSV:          869,600
Participantes:      30
Imágenes únicas:    150
Clases ADE20K:      54


In [3]:
score_rows = []
for img_id, img_data in scores_json.items():
    for entry in img_data.get('score_participant', []):
        score_rows.append({
            'ImageName':    int(img_id),
            'participante': int(entry['participant']),
            'score':        float(entry['score'])
        })

scores_df = pd.DataFrame(score_rows)
print(f'N = {len(scores_df)}  (esperado: 1,500)')
print(f'Score  media={scores_df["score"].mean():.3f}  SD={scores_df["score"].std():.3f}  '
      f'rango=[{int(scores_df["score"].min())}, {int(scores_df["score"].max())}]')
scores_df.head(3)

N = 1500  (esperado: 1,500)
Score  media=5.091  SD=1.997  rango=[1, 10]


,ImageName,participante,score
0,0,2,4.0
1,0,9,4.0
2,0,10,5.0


In [7]:
def run_lmm(class_col, label):
    """
    Ajusta LMM para un tipo de clasificación semántica.

    Parameters
    ----------
    class_col : str  — columna de clase en df
    label     : str  — nombre para mostrar en output

    Returns
    -------
    dict con: label, class_col, n_preds, n_sig, icc,
              var_part, var_res, table, result
    """
    print(f'\n{"="*72}')
    print(f'  {label}   (columna: {class_col})')
    print(f'{"="*72}')

    # ── 1. Calcular tiempo de atención por clase (Δt acumulado) ─────────────
    df_s = df.sort_values(['participante', 'ImageName', 'Time']).copy()
    df_s['_blk'] = df_s['participante'].astype(str) + '||' + df_s['ImageName'].astype(str)
    df_s['Time_next'] = df_s.groupby('_blk')['Time'].shift(-1)
    df_s['delta_t'] = (df_s['Time_next'] - df_s['Time']).clip(lower=0).fillna(0)
    df_s = df_s[df_s[class_col].notna() & (df_s[class_col].str.strip() != '')].copy()

    att = (
        df_s.groupby(['participante', 'ImageName', class_col])['delta_t']
        .sum().rename('att_sec').reset_index()
        .rename(columns={class_col: 'clase'})
    )

    # ── 2. Pivot (part × img × clases) + merge scores ───────────────────────
    pivot = att.pivot_table(
        index=['participante', 'ImageName'],
        columns='clase', values='att_sec',
        aggfunc='first', fill_value=0.0
    ).reset_index()
    pivot.columns.name = None
    model_df = pivot.merge(scores_df, on=['participante', 'ImageName'], how='inner')

    # ── 3. Renombrar columnas (fórmula patsy no admite espacios ni guiones) ──
    class_cols = [c for c in model_df.columns
                  if c not in ['participante', 'ImageName', 'score']]
    rename_map = {
        c: 'att_' + c.replace(' ', '_').replace('-', '_')
                       .replace('/', '_').replace('+', '_')
        for c in class_cols
    }
    model_df = model_df.rename(columns=rename_map)

    # ── 4. Filtrar: ≥2% filas con att > 0 y varianza > 0 ───────────────────
    min_obs   = len(model_df) * 0.02
    safe_cols = [
        c for c in rename_map.values()
        if model_df[c].std() > 1e-6 and (model_df[c] > 0).sum() >= min_obs
    ]
    print(f'  Clases únicas: {len(class_cols)}   Predictores válidos: {len(safe_cols)}')

    # ── 5. Estandarizar predictores (z-score) ───────────────────────────────
    #       Necesario: predictores en seg (0-15) vs score (1-10) → colapso var
    model_df_std = model_df.copy()
    for c in safe_cols:
        mu, sd = model_df[c].mean(), model_df[c].std()
        model_df_std[c] = (model_df[c] - mu) / sd if sd > 0 else 0.0

    # ── 6. start_params — arrancar lejos del borde var=0 ────────────────────
    #       Fix ICC=0: σ²_between - σ²_within/n como estimación inicial
    group_means   = model_df_std.groupby('participante')['score'].mean()
    n_groups      = int(group_means.shape[0])
    grand_mean    = float(model_df_std['score'].mean())
    var_between   = float(np.sum((group_means - grand_mean)**2) / (n_groups - 1))
    var_within    = float(model_df_std['score'].var(ddof=1))
    init_var_part = max(var_between - var_within / n_groups, 0.05)

    formula      = 'score ~ ' + ' + '.join(safe_cols)
    ols_coef     = smf.ols(formula, data=model_df_std).fit().params.values
    start_params = np.concatenate([ols_coef, [np.sqrt(init_var_part)]])

    # ── 7. Ajustar LMM — multi-método con criterio de gradiente ─────────────
    #       Orden: nm primero (sin gradientes, robusto en paisajes planos)
    #       Criterio: converged=True AND |grad| < 1.0 AND Part_Var > 0.01
    lmm = smf.mixedlm(
        formula=formula, data=model_df_std,
        groups=model_df_std['participante']
    )

    attempts = [
        ('nm',     {'maxiter': 10000, 'xatol': 1e-6, 'fatol': 1e-6}),
        ('lbfgs',  {'maxiter': 5000,  'gtol':  1e-5}),
        ('powell', {'maxiter': 5000,  'xtol':  1e-6, 'ftol': 1e-6}),
        ('bfgs',   {'maxiter': 5000,  'gtol':  1e-5}),
        ('cg',     {'maxiter': 5000,  'gtol':  1e-5}),
    ]

    result = best_result = None
    best_var = -1

    for method, opts in attempts:
        try:
            r     = lmm.fit(reml=True, method=method,
                            start_params=start_params, options=opts)
            var_p = float(r.cov_re.iloc[0, 0])

            # Verificar norma del gradiente si está disponible
            grad_ok = True
            grad_str = ''
            if hasattr(r, 'mle_retvals') and r.mle_retvals is not None:
                grad = r.mle_retvals.get('grad', None)
                if grad is not None:
                    gn = float(np.linalg.norm(grad))
                    grad_ok  = gn < 1.0
                    grad_str = f'  |grad|={gn:.4f}'

            print(f'  {method}: converged={r.converged}  '
                  f'Part_Var={var_p:.4f}{grad_str}')

            if var_p > best_var:
                best_var, best_result = var_p, r

            if r.converged and grad_ok and var_p > 0.01:
                result = r
                print(f'  → Seleccionado: {method}')
                break
        except Exception as e:
            print(f'  {method}: error — {e}')

    if result is None:
        result = best_result
        print(f'  → Fallback (var_part={best_var:.4f})')

    # ── 8. Extraer coeficientes ──────────────────────────────────────────────
    var_part = float(result.cov_re.iloc[0, 0])
    var_res  = float(result.scale)
    icc      = var_part / (var_part + var_res)
    ci       = result.conf_int()
    rev      = {v: k for k, v in rename_map.items()}

    rows = []
    for idx in result.params.index:
        if idx == 'Group Var':
            continue
        name = '(Intercept)' if idx == 'Intercept' else rev.get(idx, idx)
        p    = result.pvalues[idx]
        rows.append({
            'Predictor': name,
            'Coef.(β)':  round(result.params[idx], 3),
            'S.E.':      round(result.bse[idx], 3),
            'z-val':     round(result.tvalues[idx], 2),
            'p_num':     p,
            'p-val':     '<.001' if p < 0.001 else f'{p:.3f}',
            '95% CI':    f'[{ci.loc[idx,0]:.2f}, {ci.loc[idx,1]:.2f}]'
        })

    table = pd.DataFrame(rows)
    preds = table[table['Predictor'] != '(Intercept)'].copy()
    sig   = preds[preds['p_num'] < 0.05]
    print(f'  Predictores totales: {len(preds)}   '
          f'Significativos: {len(sig)} '
          f'(pos={len(sig[sig["Coef.(β)"]>0])}, '
          f'neg={len(sig[sig["Coef.(β)"] < 0])})')

    return dict(
        label=label, class_col=class_col,
        n_preds=len(safe_cols), n_sig=len(sig),
        icc=icc, var_part=var_part, var_res=var_res,
        table=table, result=result
    )

print('run_lmm() definida.')

run_lmm() definida.


In [8]:
CLASS_CONFIGS = [
    ('main_class',               'ADE20K (base)'),
    ('main_class_grouped',       'ADE20K + Grouped'),
    ('main_class_Disorder',      'ADE20K + Disorder'),
    ('main_class_GroupDisorder', 'ADE20K + Grouped + Disorder'),
]

lmm_results = {}
for class_col, label in CLASS_CONFIGS:
    lmm_results[class_col] = run_lmm(class_col, label)

print('\n✓ Todos los modelos ajustados.')


  ADE20K (base)   (columna: main_class)
  Clases únicas: 54   Predictores válidos: 27
  nm: converged=True  Part_Var=0.6646
  → Seleccionado: nm
  Predictores totales: 27   Significativos: 10 (pos=4, neg=6)

  ADE20K + Grouped   (columna: main_class_grouped)
  Clases únicas: 14   Predictores válidos: 11
  nm: converged=True  Part_Var=0.6919
  → Seleccionado: nm
  Predictores totales: 11   Significativos: 1 (pos=0, neg=1)

  ADE20K + Disorder   (columna: main_class_Disorder)
  Clases únicas: 64   Predictores válidos: 33
  nm: converged=True  Part_Var=0.7067
  → Seleccionado: nm
  Predictores totales: 33   Significativos: 14 (pos=2, neg=12)

  ADE20K + Grouped + Disorder   (columna: main_class_GroupDisorder)
  Clases únicas: 24   Predictores válidos: 18
  nm: converged=True  Part_Var=0.7152
  → Seleccionado: nm
  Predictores totales: 18   Significativos: 6 (pos=1, neg=5)

✓ Todos los modelos ajustados.


In [11]:
# ================================================================
#  PARÁMETROS
# ================================================================
MOSTRAR     = None    # 'ade20k' | 'grouped' | 'disorder' | 'grouped_disorder' | None (= los 4)
GUARDAR_PDF = True   # True → guarda PDF en la carpeta del notebook
# ================================================================

_KEY_MAP = {
    'ade20k':            'main_class',
    'grouped':           'main_class_grouped',
    'disorder':          'main_class_Disorder',
    'grouped_disorder':  'main_class_GroupDisorder',
}

def _build_text(r):
    table    = r['table']
    var_part = r['var_part']
    var_res  = r['var_res']
    icc      = r['icc']

    intercept_row = table[table['Predictor'] == '(Intercept)']
    preds = table[table['Predictor'] != '(Intercept)'].copy()
    sig   = preds[preds['p_num'] < 0.05]
    pos   = sig[sig['Coef.(β)'] > 0].sort_values('Coef.(β)', ascending=False)
    neg   = sig[sig['Coef.(β)'] < 0].sort_values('Coef.(β)', ascending=True)

    H   = f"  {'Predictor':<26} {'Coef.(β)':>9} {'S.E.':>7} {'z-val':>7} {'p-val':>7}  {'95% CI'}"
    SEP = '  ' + '─' * 74

    def fmt(row):
        return (f"  {str(row['Predictor']):<26}"
                f" {row['Coef.(β)']:>9.3f}"
                f" {row['S.E.']:>7.3f}"
                f" {row['z-val']:>7.2f}"
                f" {str(row['p-val']):>7}"
                f"  {row['95% CI']}")

    lines = []
    lines.append(f'\n{"="*78}')
    lines.append(f"  {r['label']}") #   (columna: {r['class_col']})
    lines.append(f'{"="*78}')
    lines.append(f"  Predictores incluidos: {r['n_preds']}   Significativos (p<.05): {r['n_sig']}")
    lines.append(f"\n  Table: Fixed Effects — {r['label']} (N=1500)")
    lines.append(H)
    lines.append(SEP)
    for _, row in intercept_row.iterrows():
        lines.append(fmt(row))
    if len(pos):
        lines.append('\n  Positive')
        for _, row in pos.iterrows():
            lines.append('  ' + fmt(row))
    if len(neg):
        lines.append('\n  Negative')
        for _, row in neg.iterrows():
            lines.append('  ' + fmt(row))
    lines.append(SEP)
    lines.append(f"  Random Effects            Var.      SD         ICC")
    lines.append(f"    Participant       {var_part:>9.3f}  {var_part**0.5:>6.3f}      {icc:.3f}")
    lines.append(f"    Residual          {var_res:>9.3f}  {var_res**0.5:>6.3f}")
    return '\n'.join(lines)

# ── Seleccionar resultados a mostrar ────────────────────────────────────────
if MOSTRAR is None:
    results_to_show = list(lmm_results.values())
else:
    col = _KEY_MAP.get(MOSTRAR)
    if col not in lmm_results:
        raise ValueError(f"'{MOSTRAR}' no válido. Opciones: {list(_KEY_MAP.keys())} o None")
    results_to_show = [lmm_results[col]]

# ── Construir texto ─────────────────────────────────────────────────────────
full_text = ''
for r in results_to_show:
    full_text += _build_text(r) + '\n'

if len(results_to_show) > 1:
    lines = [
        f'\n{"="*78}',
        '  Resumen comparativo',
        f'{"="*78}',
        f"  {'Clasificación':<35} {'Pred':>5} {'Sig':>5} {'Part Var':>10} {'Resid Var':>10} {'ICC':>8}",
        '  ' + '─'*72
    ]
    for r in results_to_show:
        lines.append(
            f"  {r['label']:<35} {r['n_preds']:>5} {r['n_sig']:>5}"
            f" {r['var_part']:>10.3f} {r['var_res']:>10.3f} {r['icc']:>8.3f}"
        )
    lines.append('  ' + '─'*72)
    full_text += '\n'.join(lines)

# ── Imprimir ────────────────────────────────────────────────────────────────
print(full_text)

# ── Guardar PDF ─────────────────────────────────────────────────────────────
if GUARDAR_PDF:
    label_slug = MOSTRAR if MOSTRAR else 'all'
    pdf_path   = f'lmm_results_{label_slug}.pdf'
    LINES_PER_PAGE = 55
    all_lines = full_text.split('\n')
    pages = [all_lines[i:i + LINES_PER_PAGE]
             for i in range(0, len(all_lines), LINES_PER_PAGE)]

    with PdfPages(pdf_path) as pdf:
        for page_lines in pages:
            fig, ax = plt.subplots(figsize=(11, 8.5))
            ax.axis('off')
            ax.text(0.02, 0.97, '\n'.join(page_lines),
                    transform=ax.transAxes, fontsize=8.5,
                    verticalalignment='top',
                    fontfamily='monospace', linespacing=1.4)
            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig)

    print(f'\n✓ PDF guardado: {pdf_path}  ({len(pages)} página(s))')


  ADE20K (base)
  Predictores incluidos: 27   Significativos (p<.05): 10

  Table: Fixed Effects — ADE20K (base) (N=1500)
  Predictor                   Coef.(β)    S.E.   z-val   p-val  95% CI
  ──────────────────────────────────────────────────────────────────────────
  (Intercept)                    5.091   0.155   32.79   <.001  [4.79, 5.40]

  Positive
    tree                           0.195   0.084    2.31   0.021  [0.03, 0.36]
    door                           0.182   0.051    3.57   <.001  [0.08, 0.28]
    palm                           0.130   0.056    2.32   0.020  [0.02, 0.24]
    ashcan                         0.113   0.045    2.50   0.013  [0.02, 0.20]

  Negative
    wall                          -0.370   0.154   -2.40   0.016  [-0.67, -0.07]
    sky                           -0.188   0.077   -2.43   0.015  [-0.34, -0.04]
    bridge                        -0.159   0.052   -3.06   0.002  [-0.26, -0.06]
    pole                          -0.154   0.049   -3.18   0.001  [-0

In [ ]:
import pandas as pd

df = pd.read_csv('../csv/df_final1.csv')

cols = {
    'ade20k':           'main_class',
    'grouped':          'main_class_grouped',
    'disorder':         'main_class_Disorder',
    'grouped_disorder': 'main_class_GroupDisorder',
}

# ── 1. Cuántas clases únicas tiene cada esquema ─────────────────────────────
print('='*60)
print('  Clases únicas por esquema')
print('='*60)
for key, col in cols.items():
    n = df[col].nunique()
    print(f"  {key:<20} ({col:<30})  →  {n:>3} clases")

# ── 2. Mapeo: de cada clase ADE20K a sus equivalentes agrupados ─────────────
print('\n' + '='*60)
print('  Mapeo ADE20K → grouped → disorder → grouped_disorder')
print('='*60)

mapping = (
    df[list(cols.values())]
    .drop_duplicates()
    .sort_values(list(cols.values()))
    .reset_index(drop=True)
)
mapping.columns = list(cols.keys())
print(mapping.to_string(index=False))

# ── 3. Para cada clase grouped, qué clases ADE20K contiene ──────────────────
print('\n' + '='*60)
print('  grouped → clases ADE20K que contiene')
print('='*60)
for grp, sub in (
    df.groupby('main_class_grouped')['main_class']
    .apply(lambda x: sorted(x.unique()))
    .items()
):
    print(f"\n  [{grp}]  ({len(sub)} clases)")
    for c in sub:
        print(f"    · {c}")

# ── 4. Clases nuevas en disorder (no existen en ADE20K base) ─────────────────
print('\n' + '='*60)
print('  Clases NUEVAS en disorder (no están en ADE20K base)')
print('='*60)
base    = set(df['main_class'].unique())
dis     = set(df['main_class_Disorder'].unique())
nuevas  = sorted(dis - base)
comunes = sorted(dis & base)
print(f"  ADE20K base:  {len(base)} clases")
print(f"  Disorder:     {len(dis)} clases  ({len(nuevas)} nuevas + {len(comunes)} heredadas)")
print(f"\n  Clases nuevas introducidas por Disorder:")
for c in nuevas:
    print(f"    + {c}")

# ── 5. Clases nuevas en grouped_disorder vs disorder ────────────────────────
print('\n' + '='*60)
print('  grouped_disorder vs disorder  (¿son distintos?)')
print('='*60)
gd  = set(df['main_class_GroupDisorder'].unique())
dif = sorted(gd.symmetric_difference(dis))
if not dif:
    print("  ✓ Son idénticos — mismas clases exactas")
else:
    print(f"  Diferencias: {dif}")


## CORRELAACION DE SPEARMAM: METRICAS DE ATENCION VS SCORE

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import json

# Cargar datos
df = pd.read_csv('../csv/df_final1.csv')
cov = pd.read_csv('../csv/precalculated_saliency_coverage.csv')
fix = pd.read_csv('../csv/precalculated_fixations.csv')

with open('../json/data_hololens.json') as f:
    scores_json = json.load(f)

# Construir tabla score por (participante, imagen)
score_rows = []
for img_id, img_data in scores_json.items():
    for entry in img_data.get('score_participant', []):
        score_rows.append({
            'ImageName':    int(img_id),
            'participante': int(entry['participant']),
            'score':        float(entry['score'])
        })
scores_df = pd.DataFrame(score_rows)

# Calcular entropy por (participante, imagen)
def shannon_entropy(series):
    p = series / series.sum()
    p = p[p > 0]
    return float(-np.sum(p * np.log2(p)))

att = (
    df.assign(
        _blk=df['participante'].astype(str) + '||' + df['ImageName'].astype(str)
    )
    .assign(Time_next=lambda x: x.groupby('_blk')['Time'].shift(-1))
    .assign(delta_t=lambda x: (x['Time_next'] - x['Time']).clip(lower=0).fillna(0))
)
att_by_class = (
    att.groupby(['participante', 'ImageName', 'main_class'])['delta_t']
    .sum().reset_index()
)
entropy_df = (
    att_by_class.groupby(['participante', 'ImageName'])['delta_t']
    .apply(shannon_entropy).reset_index()
    .rename(columns={'delta_t': 'entropy'})
)
n_fix = (
    att_by_class.groupby(['participante', 'ImageName'])['delta_t']
    .count().reset_index()
    .rename(columns={'delta_t': 'n_classes_attended'})
)
total_time = (
    att_by_class.groupby(['participante', 'ImageName'])['delta_t']
    .sum().reset_index()
    .rename(columns={'delta_t': 'total_att_time'})
)

analysis = scores_df \
    .merge(entropy_df,   on=['participante','ImageName'], how='left') \
    .merge(n_fix,        on=['participante','ImageName'], how='left') \
    .merge(total_time,   on=['participante','ImageName'], how='left')

# Correlaciones de Spearman
metrics = ['entropy', 'n_classes_attended', 'total_att_time']
print('='*55)
print('  Spearman: métricas de atención vs. score')
print('='*55)
print(f"  {'Métrica':<25} {'ρ':>8} {'p-val':>10}")
print('  ' + '─'*45)
for m in metrics:
    sub = analysis[[m, 'score']].dropna()
    rho, pval = stats.spearmanr(sub[m], sub['score'])
    sig = '***' if pval < .001 else '**' if pval < .01 else '*' if pval < .05 else ''
    print(f"  {m:<25} {rho:>8.3f} {pval:>10.4f} {sig}")


## Kruskal-Wallis + Dunn: ¿difiere el score entre clases más miradas

In [ ]:
!pip install scikit_posthocs

In [ ]:
from scipy.stats import kruskal
import scikit_posthocs as sp   # pip install scikit-posthocs

# Clase dominante por observación (la más mirada en esa imagen)
dominant = (
    att_by_class.sort_values('delta_t', ascending=False)
    .groupby(['participante','ImageName'])
    .first()
    .reset_index()[['participante','ImageName','main_class']]
    .rename(columns={'main_class': 'dominant_class'})
)
kw_df = scores_df.merge(dominant, on=['participante','ImageName'])

# Solo clases con ≥ 30 observaciones
counts = kw_df['dominant_class'].value_counts()
valid  = counts[counts >= 30].index
kw_df  = kw_df[kw_df['dominant_class'].isin(valid)]

groups = [g['score'].values for _, g in kw_df.groupby('dominant_class')]
stat, pval = kruskal(*groups)
print(f'\nKruskal-Wallis  H={stat:.3f}  p={pval:.4f}')

# Post-hoc Dunn (corrección Bonferroni)
dunn = sp.posthoc_dunn(kw_df, val_col='score', group_col='dominant_class', p_adjust='bonferroni')
print('\nPost-hoc Dunn (solo pares significativos p<.05):')
for col in dunn.columns:
    for idx in dunn.index:
        if idx < col and dunn.loc[idx, col] < 0.05:
            print(f"  {idx} vs {col}:  p={dunn.loc[idx,col]:.4f}")


## ICC inter-rater entre participantes (confiabilidad del score) AHJORA krippendorff

In [ ]:
!pip install krippendorff

In [ ]:
import subprocess, sys
# Verificar que krippendorff está instalado en este kernel
result = subprocess.run([sys.executable, '-m', 'pip', 'show', 'krippendorff'],
                        capture_output=True, text=True)
if 'Version' in result.stdout:
    print('✓', result.stdout.split('\n')[1])  # muestra versión
else:
    # Instalar directamente en este kernel si falta
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'krippendorff', '-q'])
    print('✓ krippendorff instalado')


In [ ]:
import pandas as pd
import numpy as np
import json
import krippendorff
from scipy import stats

# ── Cargar scores ────────────────────────────────────────────────────────────
with open('../json/data_hololens.json') as f:
    scores_json = json.load(f)

score_rows = []
for img_id, img_data in scores_json.items():
    for entry in img_data.get('score_participant', []):
        score_rows.append({
            'ImageName':    int(img_id),
            'participante': int(entry['participant']),
            'score':        float(entry['score'])
        })
scores_df = pd.DataFrame(score_rows)

print(f'N={len(scores_df)} | Imágenes={scores_df["ImageName"].nunique()} | '
      f'Participantes={scores_df["participante"].nunique()}')
print(f'Ratings por imagen: {scores_df.groupby("ImageName")["participante"].count().unique()}')

# ── Tabla wide: filas=raters (participantes), columnas=items (imágenes) ──────
# NaN donde el participante NO calificó esa imagen
wide = scores_df.pivot(
    index='participante', columns='ImageName', values='score'
)
print(f'\nTabla wide: {wide.shape[0]} raters × {wide.shape[1]} items')
print(f'Celdas con dato: {wide.notna().sum().sum()} / {wide.size}')

# ── 1. Krippendorff's Alpha (maneja NaN nativamente) ─────────────────────────
# metric='ordinal' para escala 1-10 ordinal
# metric='interval' si la escala es continua/intervalar
alpha_ord = krippendorff.alpha(
    reliability_data=wide.values,
    level_of_measurement='ordinal'
)
alpha_int = krippendorff.alpha(
    reliability_data=wide.values,
    level_of_measurement='interval'
)

print('\n' + '='*55)
print('  Krippendorff\'s Alpha — Confiabilidad inter-rater')
print('='*55)
print(f'  α ordinal  = {alpha_ord:.3f}')
print(f'  α interval = {alpha_int:.3f}')
print()
for alpha, tipo in [(alpha_ord, 'ordinal'), (alpha_int, 'interval')]:
    if alpha >= 0.80:
        interp = 'Excelente'
    elif alpha >= 0.667:
        interp = 'Aceptable (publicable)'
    elif alpha >= 0.50:
        interp = 'Moderada'
    else:
        interp = 'Pobre'
    print(f'  α {tipo:<9} → {interp}')

print('\n  Referencia: Krippendorff (2004)')
print('  α ≥ 0.80 → excelente | α ≥ 0.667 → aceptable | α < 0.50 → pobre')

# ── 2. Estadísticas descriptivas del score ───────────────────────────────────
print('\n' + '='*55)
print('  Descriptivos del score de seguridad')
print('='*55)
desc = scores_df['score'].describe()
skew = scores_df['score'].skew()
kurt = scores_df['score'].kurtosis()
print(f'  Media:    {desc["mean"]:.3f}')
print(f'  SD:       {desc["std"]:.3f}')
print(f'  Mediana:  {desc["50%"]:.1f}')
print(f'  Rango:    [{int(desc["min"])}, {int(desc["max"])}]')
print(f'  Asimetría:{skew:.3f}  (>0 cola derecha, <0 cola izquierda)')
print(f'  Curtosis: {kurt:.3f}')

# ── 3. Varianza del score por imagen (consenso vs controversia) ───────────────
score_stats = scores_df.groupby('ImageName')['score'].agg(
    mean='mean', std='std',
    cv=lambda x: x.std() / x.mean()
).reset_index()

print('\n' + '='*55)
print('  Variabilidad del score por imagen')
print('='*55)
print(f'  SD media por imagen:  {score_stats["std"].mean():.3f}')
print(f'  CV medio por imagen:  {score_stats["cv"].mean():.3f}')
print(f'\n  Top 5 más CONSENSUADAS (menor SD):')
for _, r in score_stats.nsmallest(5, 'std').iterrows():
    print(f'    Img {int(r.ImageName):>3}  mean={r["mean"]:.2f}  SD={r["std"]:.2f}')
print(f'\n  Top 5 más CONTROVERTIDAS (mayor SD):')
for _, r in score_stats.nlargest(5, 'std').iterrows():
    print(f'    Img {int(r.ImageName):>3}  mean={r["mean"]:.2f}  SD={r["std"]:.2f}')


##  Análisis de fijaciones

In [ ]:
fix_df = pd.read_csv('../csv/precalculated_fixations.csv')
print(fix_df.columns.tolist())  # ver columnas disponibles

# Hipótesis: las fijaciones sobre clases "seguras" duran más que las "inseguras"
safe_classes   = ['tree', 'door', 'palm']
unsafe_classes = ['wall', 'bridge', 'pole', 'mountain']

fix_merged = fix_df.merge(
    df[['participante','ImageName','Time','main_class']].drop_duplicates(),
    on=['participante','ImageName'], how='left'
)

safe_dur   = fix_merged[fix_merged['main_class'].isin(safe_classes)]['duration']
unsafe_dur = fix_merged[fix_merged['main_class'].isin(unsafe_classes)]['duration']

stat, pval = stats.mannwhitneyu(safe_dur.dropna(), unsafe_dur.dropna(), alternative='two-sided')
print(f'\nMann-Whitney U: safe vs unsafe classes')
print(f'  U={stat:.0f}  p={pval:.4f}')
print(f'  Median safe:   {safe_dur.median():.3f} s')
print(f'  Median unsafe: {unsafe_dur.median():.3f} s')


## Varianza del score por imagen (¿hay imágenes más controversiales?)

In [ ]:
# Coeficiente de variación por imagen
score_stats = scores_df.groupby('ImageName')['score'].agg(
    mean='mean', std='std', cv=lambda x: x.std()/x.mean()
).reset_index()

print('\nTop 5 imágenes más CONTROVERSIALES (mayor CV):')
print(score_stats.nlargest(5, 'cv')[['ImageName','mean','std','cv']].round(3).to_string(index=False))

print('\nTop 5 imágenes más CONSENSUADAS (menor CV):')
print(score_stats.nsmallest(5, 'cv')[['ImageName','mean','std','cv']].round(3).to_string(index=False))
